## Feature engineering : Which features to train your model on and why?

In [1]:
import json

import joblib
import pandas as pd

from sklearn.preprocessing import OneHotEncoder

from src.utils.paths import MODELS_DIR, PROCESSED_DIR

pd.set_option("display.max_columns", None)

In [2]:
train_df = pd.read_csv(PROCESSED_DIR / "cleaning_train.csv")
eval_df = pd.read_csv(PROCESSED_DIR / "cleaning_eval.csv")

In [3]:
print("train:", train_df.shape, "| eval:", eval_df.shape)

train: (4930, 21) | eval: (1056, 21)


In [4]:
train_df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,6918-UMQCG,Female,0,No,No,5,Yes,No,Fiber optic,No,No,No,No,Yes,No,Month-to-month,No,Electronic check,80.20,384.25,0
1,4140-MUHUG,Female,1,No,No,3,Yes,No,Fiber optic,No,No,Yes,No,Yes,No,Month-to-month,Yes,Electronic check,86.85,220.95,1
2,5570-PTWEH,Female,0,Yes,No,3,Yes,Yes,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Credit card (automatic),75.15,216.75,1
3,2054-PJOCK,Female,0,No,No,60,Yes,Yes,DSL,No,No,Yes,Yes,Yes,Yes,One year,No,Credit card (automatic),80.55,4847.05,0
4,4315-MURBD,Female,0,No,No,12,Yes,No,Fiber optic,Yes,No,No,Yes,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),98.90,1120.95,1


## 0. Quick recap: what the columns mean and what the EDA told us

This notebook uses words from the EDA, so here is the short version before we build anything.

| Group | Columns | Plain meaning |
|---|---|---|
| Customer | `SeniorCitizen`, `Partner`, `Dependents`, `gender` | who the customer is |
| Time | `tenure` | how many **months** they have been a customer |
| Phone | `PhoneService`, `MultipleLines` | do they have a phone line, and more than one |
| Internet | `InternetService` | `DSL`, `Fiber optic` or `No` |
| **Add-ons** | `OnlineSecurity`, `OnlineBackup`, `DeviceProtection`, `TechSupport`, `StreamingTV`, `StreamingMovies` | **optional extras** a customer pays for on top of their internet. You can only have them if you have internet |
| Billing | `Contract`, `PaperlessBilling`, `PaymentMethod` | how long they committed for and how they pay. `Contract` is `Month-to-month`, `One year` or `Two year`. Two payment methods are "(automatic)": bank transfer and credit card |
| Money | `MonthlyCharges`, `TotalCharges` | the current monthly bill, and everything paid so far |
| Target | `Churn` | 1 = the customer left, 0 = stayed |

**The EDA findings that drive this notebook** (the next cell recomputes them on the train set so you can see them):

1. **New customers leave most.** Churn is highest in the first year of `tenure`.
2. **Month-to-month contracts leave most.** A one- or two-year contract keeps people.
3. **Customers with no internet almost never leave.** Fiber customers leave the most.
4. **The more add-ons, the fewer cancellations.** Add-ons tie a customer to the company.
5. **Manual payers leave more.** Electronic check is the worst; automatic payment is the best.
6. **`gender` shows no signal.**

Total charges ~ tenure X monthly charges


In [5]:
def churn_by(df, col):
    return df.groupby(col, observed=True)["Churn"].agg(customers="count", churn_rate="mean").round(3)

print(f"Overall churn rate (train): {train_df['Churn'].mean():.1%}")

tenure_band = pd.cut(train_df["tenure"], [-1, 12, 24, 48, 72], labels=["0-12", "13-24", "25-48", "49-72"])
print("\nBy tenure band (months)")
display(churn_by(train_df, tenure_band))

for col in ["Contract", "InternetService", "PaymentMethod", "gender"]:
    print(f"\nBy {col}")
    display(churn_by(train_df, col))

print("\nBy TechSupport, internet customers only")
display(churn_by(train_df[train_df["InternetService"] != "No"], "TechSupport"))

Overall churn rate (train): 26.5%

By tenure band (months)


,customers,churn_rate
tenure,,
0-12,1537,0.477
13-24,707,0.273
25-48,1118,0.199
49-72,1568,0.101



By Contract


,customers,churn_rate
Contract,,
Month-to-month,2704,0.429
One year,1034,0.109
Two year,1192,0.030



By InternetService


...

,customers,churn_rate
PaymentMethod,,
Bank transfer (automatic),1086,0.163
Credit card (automatic),1067,0.147
Electronic check,1654,0.456
Mailed check,1123,0.195



By gender


,customers,churn_rate
gender,,
Female,2452,0.263
Male,2478,0.268



By TechSupport, internet customers only


,customers,churn_rate
TechSupport,,
No,2411,0.419
Yes,1451,0.154


## 1. Drop what we should not use

| Column | Decision | Why |
|---|---|---|
| `customerID` | drop from features | an identifier, no signal. We keep it aside for batch output |
| `gender` | drop | no signal in the EDA (26.8% vs 26.3% churn) and no reason to use it |
| `TotalCharges` | drop | it is almost `tenure x MonthlyCharges`, so it adds no new information and is 0.83 correlated with `tenure` |

Everything else stays. `SeniorCitizen` stays but we will review it for fairness before deploying.

In [6]:
ID_COL, TARGET = "customerID", "Churn"

DROP_COLS = ['gender', 'TotalCharges']

In [7]:
train_df.OnlineSecurity.value_counts()

OnlineSecurity
No                     2439
Yes                    1423
No internet service    1068
Name: count, dtype: int64

In [10]:
train_df.OnlineBackup.value_counts()

OnlineBackup
No                     2142
Yes                    1720
No internet service    1068
Name: count, dtype: int64

In [11]:
addon_cols = ["OnlineSecurity", "OnlineBackup", "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]

In [12]:
def simplify_services(df):
    df = df.copy()
    service_cols = addon_cols + ["MultipleLines"]
    df[service_cols] = df[service_cols].replace({"No internet service": "No", "No phone service": "No"})
    return df

print(train_df["OnlineSecurity"].value_counts().to_dict())
print(simplify_services(train_df)["OnlineSecurity"].value_counts().to_dict())

{'No': 2439, 'Yes': 1423, 'No internet service': 1068}
{'No': 3507, 'Yes': 1423}


## 3. New features, each one from an EDA finding

| New feature | How | Why (from the EDA) |
|---|---|---|
| `is_first_year` | `tenure <= 12` | churn is highest in the first year |
| `has_internet` | `InternetService != "No"` | customers without internet almost never churn (7%) |
| `addon_count` | number of add-ons that are `Yes` | more add-ons, fewer cancellations |
| `autopay` | payment method contains "automatic" | automatic payers churn less than half as much |
| `spend_per_service` | `MonthlyCharges / number of services` | paying a lot for little is a risk. A model cannot easily build a ratio by itself |

We tried one idea and **dropped it**: "average price paid vs current price". The average price paid is almost the same as `MonthlyCharges` (correlation about 1.0), so it adds nothing.

In [13]:
def add_features(df):
    df = df.copy()
    df["is_first_year"] = (df["tenure"] <= 12).astype(int)
    df["has_internet"] = (df["InternetService"] != "No").astype(int)
    df["addon_count"] = df[addon_cols].eq("Yes").sum(axis=1)
    df["autopay"] = df["PaymentMethod"].str.contains("automatic").astype(int)

    n_services = (df["PhoneService"] == "Yes").astype(int) + df["has_internet"] + df["addon_count"]
    df["spend_per_service"] = df["MonthlyCharges"] / n_services.clip(lower=1)
    return df

In [14]:
check = add_features(simplify_services(train_df))

for col in ["is_first_year", "has_internet", "autopay"]:
    display(check.groupby(col)["Churn"].agg(customers="count", churn_rate="mean"))

,customers,churn_rate
is_first_year,,
0,3393,0.169467
1,1537,0.476903


,customers,churn_rate
has_internet,,
0,1068,0.069288
1,3862,0.319524


,customers,churn_rate
autopay,,
0,2777,0.350738
1,2153,0.155132


In [15]:
# add-ons only make sense for customers who have internet
(check[check["has_internet"] == 1]
 .groupby("addon_count")["Churn"].agg(customers="count", churn_rate="mean"))

,customers,churn_rate
addon_count,,
0,470,0.510638
1,658,0.466565
2,733,0.371078
3,773,0.274256
4,612,0.230392
5,399,0.130326
6,217,0.046083


In [16]:
check.groupby(pd.qcut(check["spend_per_service"], 5), observed=True)["Churn"].agg(customers="count", churn_rate="mean")

,customers,churn_rate
spend_per_service,,
"(8.649000000000001, 13.824]",986,0.105477
"(13.824, 17.5]",992,0.238911
"(17.5, 20.0]",991,0.231080
"(20.0, 23.8]",978,0.319018
"(23.8, 38.95]",983,0.433367


In [17]:
yes_no_cols = ["Partner", "Dependents", "PhoneService", "MultipleLines", "PaperlessBilling"] + addon_cols

def prepare(df):
    df = add_features(simplify_services(df))
    df[yes_no_cols] = df[yes_no_cols].apply(lambda s: s.map({"Yes": 1, "No": 0}))
    return df.drop(columns=DROP_COLS)

train_ready = prepare(train_df)
eval_ready = prepare(eval_df)
train_ready.head(3)

,customerID,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,Churn,is_first_year,has_internet,addon_count,autopay,spend_per_service
0,6918-UMQCG,0,0,0,5,1,0,Fiber optic,0,0,0,0,1,0,Month-to-month,0,Electronic check,80.20,0,1,1,1,0,26.733333
1,4140-MUHUG,1,0,0,3,1,0,Fiber optic,0,0,1,0,1,0,Month-to-month,1,Electronic check,86.85,1,1,1,2,0,21.712500
2,5570-PTWEH,0,1,0,3,1,1,Fiber optic,0,0,0,0,0,0,Month-to-month,1,Credit card (automatic),75.15,1,1,1,0,1,37.575000


## 5. One-hot encode the last three text columns

`Contract`, `InternetService` and `PaymentMethod` have 3 or 4 values each, so **one-hot encoding** is enough: one 0/1 column per value.

The reference housing project needed target and frequency encoding because `city` has hundreds of values.
We do not have that problem, so we use the simplest method that works.

**Leakage rule:** the encoder *learns* the list of categories, so we fit it on **train only**. Eval is only transformed.
`handle_unknown="ignore"` means a new category in future data becomes all zeros instead of crashing.

In [18]:
cat_cols = ["Contract", "InternetService", "PaymentMethod"]

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder.fit(train_ready[cat_cols])          # learn the categories from TRAIN only

,"sparse_output sparse_output: bool, default=TrueWhen ``True``, it returns a SciPy sparse matrix/arrayin ""Compressed Sparse Row"" (CSR) format... versionadded:: 1.2 `sparse` was renamed to `sparse_output`",False
,"handle_unknown handle_unknown: {'error', 'ignore', 'infrequent_if_exist', 'warn'}, default='error'Specifies the way unknown categories are handled during :meth:`transform`.- 'error' : Raise an error if an unknown category is present during transform.- 'ignore' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will be all zeros. In the inverse transform, an unknown category will be denoted as None.- 'infrequent_if_exist' : When an unknown category is encountered during transform, the resulting one-hot encoded columns for this feature will map to the infrequent category if it exists. The infrequent category will be mapped to the last position in the encoding. During inverse transform, an unknown category will be mapped to the category denoted `'infrequent'` if it exists. If the `'infrequent'` category does not exist, then :meth:`transform` and :meth:`inverse_transform` will handle an unknown category as with `handle_unknown='ignore'`. Infrequent categories exist based on `min_frequency` and `max_categories`. Read more in the :ref:`User Guide <encoder_infrequent_categories>`.- 'warn' : When an unknown category is encountered during transform a warning is issued, and the encoding then proceeds as described for `handle_unknown=""infrequent_if_exist""`... versionchanged:: 1.1 `'infrequent_if_exist'` was added to automatically handle unknown categories and infrequent categories... versionadded:: 1.6 The option `""warn""` was added in 1.6.",'ignore'
,"categories categories: 'auto' or a list of array-like, default='auto'Categories (unique values) per feature:- 'auto' : Determine categories automatically from the training data.- list : ``categories[i]`` holds the categories expected in the ith column. The passed categories should not mix strings and numeric values within a single feature, and should be sorted in case of numeric values.The used categories can be found in the ``categories_`` attribute... versionadded:: 0.20",'auto'
,"drop drop: {'first', 'if_binary'} or an array-like of shape (n_features,), default=NoneSpecifies a methodology to use to drop one of the categories perfeature. This is useful in situations where perfectly collinearfeatures cause problems, such as when feeding the resulting datainto an unregularized linear regression model.However, dropping one category breaks the symmetry of the originalrepresentation and can therefore induce a bias in downstream models,for instance for penalized linear classification or regression models.- None : retain all features (the default).- 'first' : drop the first category in each feature. If only one category is present, the feature will be dropped entirely.- 'if_binary' : drop the first category in each feature with two categories. Features with 1 or more than 2 categories are left intact.- array : ``drop[i]`` is the category in feature ``X[:, i]`` that should be dropped.When `max_categories` or `min_frequency` is configured to groupinfrequent categories, the dropping behavior is handled after thegrouping... versionadded:: 0.21 The parameter `drop` was added in 0.21... versionchanged:: 0.23 The option `drop='if_binary'` was added in 0.23... versionchanged:: 1.1 Support for dropping infrequent categories.",None
,"dtype dtype: number type, default=np.float64Desired dtype of output.",<class 'numpy.float64'>
,"min_frequency min_frequency: int or float, default=NoneSpecifies the minimum frequency below which a category will beconsidered infrequent.- If `int`, categories with a smaller cardinality will be considered infrequent.- If `float`, categories with a smaller cardinality than `min_frequency * n_samples` will be considered infrequent... versionadded:: 1.1 Read more in the :ref:`User Guide <encoder_infreque

In [19]:
def to_features(df):
    encoded = pd.DataFrame(encoder.transform(df[cat_cols]),
                           columns=encoder.get_feature_names_out(cat_cols), index=df.index)
    return pd.concat([df.drop(columns=cat_cols), encoded], axis=1)

train_features = to_features(train_ready)
eval_features = to_features(eval_ready)
train_features.head(3)

,customerID,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,PaperlessBilling,MonthlyCharges,Churn,is_first_year,has_internet,addon_count,autopay,spend_per_service,Contract_Month-to-month,Contract_One year,Contract_Two year,InternetService_DSL,InternetService_Fiber optic,InternetService_No,PaymentMethod_Bank transfer (automatic),PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check
0,6918-UMQCG,0,0,0,5,1,0,0,0,0,0,1,0,0,80.20,0,1,1,1,0,26.733333,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,4140-MUHUG,1,0,0,3,1,0,0,0,1,0,1,0,1,86.85,1,1,1,2,0,21.712500,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
2,5570-PTWEH,0,1,0,3,1,1,0,0,0,0,0,0,1,75.15,1,1,1,0,1,37.575000,1.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0


In [20]:
train_features.columns

Index(['customerID', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
       'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
       'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies',
       'PaperlessBilling', 'MonthlyCharges', 'Churn', 'is_first_year',
       'has_internet', 'addon_count', 'autopay', 'spend_per_service',
       'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year',
       'InternetService_DSL', 'InternetService_Fiber optic',
       'InternetService_No', 'PaymentMethod_Bank transfer (automatic)',
       'PaymentMethod_Credit card (automatic)',
       'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check'],
      dtype='str')

In [21]:
ID_COL,TARGET, DROP_COLS

('customerID', 'Churn', ['gender', 'TotalCharges'])

In [23]:
feature_cols = [ c for c in train_features.columns if c not in (ID_COL, TARGET)]

In [24]:
assert list(train_features.columns) == list(eval_features.columns)          # same columns, same order
assert train_features.isna().sum().sum() == 0 and eval_features.isna().sum().sum() == 0
assert train_features[feature_cols].select_dtypes(exclude="number").empty    # everything is numeric
assert TARGET not in feature_cols                                           # the answer is not a feature

print(len(feature_cols), "features:")
print(feature_cols)

29 features:
['SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'MonthlyCharges', 'is_first_year', 'has_internet', 'addon_count', 'autopay', 'spend_per_service', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'PaymentMethod_Bank transfer (automatic)', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


In [25]:
train_features.to_csv(PROCESSED_DIR / "features_train.csv", index=False)
eval_features.to_csv(PROCESSED_DIR / "features_eval.csv", index=False)

MODELS_DIR.mkdir(exist_ok=True)
joblib.dump(encoder, MODELS_DIR / "encoder.joblib")
(MODELS_DIR / "feature_columns.json").write_text(json.dumps(feature_cols, indent=2))

print("✅ Features saved.")

✅ Features saved.
